# Questionário 5

Aline da Silva Pereira - 6485698

In [1]:
from collections import Counter
from collections.abc import Sequence
from pathlib import Path

In [35]:
import networkx as nx
import numpy as np
import pandas as pd
from networkx import Graph

In [3]:
DATA_DIR = Path("../data").resolve()

In [34]:
def node_degrees(G: Graph) -> dict[int, int]:
    return {i: G.degree(i) for i in G.nodes}

In [5]:
def moment(data: Sequence[float], *, order: int = 1) -> float:
    return np.mean(np.pow(data, order))

In [6]:
def graph_complexity(data: Sequence[float]) -> float:
    return moment(data, order=2) / moment(data, order=1)

In [7]:
def calculate_distribution(data: Sequence[float]) -> dict[float, float]:
    counts = Counter(data)
    n = len(data)
    return {k: nk / n for k, nk in counts.items()}

In [8]:
def shannon_entopy(data: Sequence[float]) -> float:
    distribution = calculate_distribution(data)
    probs = np.array([p for p in distribution.values()])
    return -probs.dot(np.log2(probs))

In [9]:
def normalize_graph(G: Graph) -> Graph:
    G = G.to_undirected()
    G.remove_edges_from(nx.selfloop_edges(G))
    Gcc = sorted(nx.connected_components(G), key=len, reverse=True)
    G = G.subgraph(Gcc[0])
    G = nx.convert_node_labels_to_integers(G, first_label=0)
    return G

## Questão 1

In [11]:
G: Graph = nx.read_edgelist(DATA_DIR / "USairport500.txt")
G = normalize_graph(G)

In [25]:
ev_centrality = np.array([*nx.eigenvector_centrality(G).values()])


In [26]:
print(f"{ev_centrality.mean() = :.2f}")

ev_centrality.mean() = 0.02


## Questão 2

In [28]:
G: Graph = nx.read_edgelist(DATA_DIR / "hamsterster.txt")
G = normalize_graph(G)

In [33]:
degrees = node_degrees(G)
bw_centrality = nx.betweenness_centrality(G)

In [ ]:
data = (
    pd.Series(degrees)
    .to_frame("degree")
    .join(pd.Series(bw_centrality).to_frame("centrality"))
)

In [42]:
data.corr()

,degree,centrality
degree,1.000000,0.824244
centrality,0.824244,1.000000


## Questão 3

In [45]:
G: Graph = nx.read_edgelist(DATA_DIR / "jazz.txt")
G = normalize_graph(G)

In [47]:
closeness = nx.closeness_centrality(G)
k_core = nx.core_number(G)

In [48]:
data = (
    pd.Series(closeness)
    .to_frame("degree")
    .join(pd.Series(k_core).to_frame("k_core"))
)

In [50]:
data.corr(method="spearman")

,degree,k_core
degree,1.00000,0.73534
k_core,0.73534,1.00000
